# 15 — Controlled Field-only versus Mixed-domain Fine-tuning

## 1. Research question

**Does adding laboratory rehearsal improve Field generalization when Field-only and Mixed fine-tuning receive identical Field-image exposure and optimizer updates?**

Historical Field-only adaptation used 370 Field images per epoch, while Mixed training sampled 185 Field plus 185 Lab images. Field-only used full batches of 32 (ResNet) or 16 (ViT); Mixed used half as many Field images per batch. Field-only and Mixed also stopped and selected checkpoints independently, rewound to each phase's best checkpoint, and used a combined mean loss that scaled the Field contribution by ½. The final historical geometric ResNet comparison additionally used a geometric initialization for Field-only and a Lab initialization for Mixed. These comparisons motivate a controlled experiment; their results do not answer this question directly.

This notebook is prepared for **Google Colab CUDA execution**. New training and evaluation results are pending. No controlled model has been trained locally, and no result is claimed until the cells below execute on Colab.

## 2. Controlled experiment design

| Component | Field-only | Mixed-domain |
| --- | --- | --- |
| Initialization | Recorded Lab baseline | Exact same checkpoint and tensor state |
| Field examples per epoch | All 370 verified adaptation rows | Same 370 rows, order and augmented tensors |
| Objective per update | Mean Field cross-entropy | Mean Field cross-entropy + mean Lab cross-entropy |
| Lab loss weight | — | λ = 1, fixed before evaluation |
| Lab presentations per epoch | 0 | 370, drawn without replacement within each epoch |
| Optimizer | AdamW | Same family, learning rates and update count |
| Schedule | 2 head epochs + 6 final-block epochs | Same; no early stopping or phase rewind |
| Checkpoint selection | Shared epoch selected from Field Dev | Same epoch as Field-only |

The original 476-image Field Train includes the 92 Field Dev and 14 quarantined rows. Both conditions use the existing **370-image `field_adapt_train.csv`**, preserving all established exclusions. The fixed λ = 1 gives equal weight to the two mean losses and preserves the historical equal-domain emphasis without the historical factor of ½ on the Field gradient. It is a design choice, not a tuned optimum; no held-out score chooses it. Each paired update reuses one augmented Field tensor; Mixed adds a separately normalized Lab loss through gradient accumulation before its one optimizer step. BatchNorm running statistics remain fixed.

Learning rates are 10⁻⁴ for the classifier and 10⁻⁵ for the unfrozen final block, with weight decay 10⁻⁴. ResNet18 uses 16 Field images per update; ViT-Small uses 8. The final batch may be smaller. The optimizer is reset once at the phase transition in both conditions. Training follows the existing mild crop/flip/jitter policy and model-specific normalization.

**Selection rule:** maximize the mean Field Dev macro-F1 across the two conditions over epochs 0–8, retaining the earliest tie. This selects the same epoch and hence the same exposure/update count for both checkpoints. Independently choosing each model's best epoch would break that exposure control. The symmetric joint rule couples selection across conditions and is part of the experimental design.

Mixed performs additional computation and receives additional Lab information; this is an exposure- and update-matched comparison, not a compute-matched comparison. Seeds are fixed at 42, 43 and 44. A one-seed pilot may instead be declared before initialization; never choose the best seed from observed scores.

## 3. Google Colab setup

Select **Runtime → Change runtime type → GPU** before executing code. Start with ResNet18. Data, checkpoints and outputs live on Google Drive. Code and canonical splits are fetched from GitHub; no code ZIP is needed.

Upload only the private folder `colab_assets_v1`, including all nested files, into:

```
MyDrive/plant-disease-diagnosis/controlled_field_mixed_assets/
```

The asset folder contains the existing HF source rows, canonical Lab baselines and a hash manifest. These assets are excluded from Git. Upload this notebook separately to Colab, or open the published notebook from GitHub. See `docs/notebook15_colab_guide.md` in the repository.

In code cell 3, set `GIT_REF` to the branch containing Notebook 15 (default `main`) or an exact commit SHA. The helper module and splits must be committed and pushed to that ref first. Code cell 4 fetches that revision without overwriting local changes and prints its resolved commit. Save that SHA and use it as `GIT_REF` when resuming in a fresh session. A private repository requires Git authentication in Colab.

**Execution order (code cells):** 1 CUDA → 2 dependencies → 3 Drive/paths → 4 GitHub code → 5 experiment configuration → 6 schedule assertions → 7 CUDA smoke test. Only then execute code cell 8 (seed 42), cell 9 (remaining declared seeds), cell 10 (Dev curves) and cell 11 (sealed Field Test evaluation). The remaining cells report errors, figures and findings. Cell titles are shown directly in the code.


In [ ]:
# Code cell 1 — CUDA gate.
# CUDA gate runs before any install, repository setup or training.
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Use Google Colab: Runtime → Change runtime type → GPU. Do not run training on the local Mac.')
print(f'CUDA GPU: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}')

In [ ]:
# Code cell 2 — Install dependencies.
# Keep Colab's matched torch/torchvision CUDA installation.
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'timm==1.0.26', 'datasets==5.1.0', 'scikit-learn', 'pandas', 'matplotlib'])

In [ ]:
# Code cell 3 — Mount Drive and configure persistent paths.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

GIT_URL = 'https://github.com/thutrang147/plant-disease-diagnosis.git'
GIT_REF = 'main'  # Branch containing Notebook 15, or exact commit SHA for resume.
REPO = Path('/content/plant-disease-diagnosis-controlled15')
DRIVE_PROJECT = Path('/content/drive/MyDrive/plant-disease-diagnosis')
ASSET_ROOT = DRIVE_PROJECT / 'controlled_field_mixed_assets/colab_assets_v1'
OUTPUT_ROOT = DRIVE_PROJECT / 'outputs/controlled_field_mixed'
if not OUTPUT_ROOT.is_relative_to(Path('/content/drive/MyDrive')):
    raise RuntimeError('Use Google Drive persistent storage for this Colab workflow.')
if not (ASSET_ROOT / 'asset_manifest.json').is_file():
    raise FileNotFoundError(
        f'Missing {ASSET_ROOT / "asset_manifest.json"}. Upload the complete '
        'colab_assets_v1 folder to ASSET_ROOT, or correct ASSET_ROOT. '
        'Data and checkpoints are excluded from GitHub; no code ZIP is required.'
    )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Code cell 4 — Clone/fetch code and canonical splits from GitHub.
import subprocess
import sys
import os
import importlib
import json

def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if not REPO.exists():
    subprocess.check_call(['git', 'clone', GIT_URL, str(REPO)])
if not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} is not a Git checkout. Choose a fresh REPO path.')
if git('remote', 'get-url', 'origin') != GIT_URL:
    raise RuntimeError('Existing checkout has a different origin. Choose a fresh REPO path.')
if git('status', '--porcelain'):
    raise RuntimeError('Checkout has local changes. Save them and choose a fresh REPO path.')
subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', GIT_REF])
subprocess.check_call(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'])
GIT_COMMIT = git('rev-parse', 'HEAD')
required = ['src/controlled_field_mixed.py', 'data/splits/label_map.json']
required += [f'data/splits/{name}.csv' for name in
             ('field_adapt_train', 'field_dev', 'field_test', 'lab_train')]
missing = [name for name in required if not (REPO / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'GitHub ref {GIT_REF!r} is missing: {missing}. Commit and push Notebook 15 '
        'and its helper/splits, then set GIT_REF to that branch or commit and rerun.'
    )
print('GitHub commit:', GIT_COMMIT)
print('For resume, set GIT_REF to this commit SHA before rerunning setup.')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
from src.controlled_field_mixed import (
    Assets, CLASSES, CONDITIONS, default_config, epoch_plan, open_experiment,
    smoke_test, train_pair, validate_training, evaluate_heldout,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})


## 4. Dataset and checkpoint verification

The asset loader verifies file SHA-256, the exact split files, original source row/class mappings, label order, checkpoint identity and the preserved duplicate screen. A missing or incompatible asset stops execution; no alternative weights or datasets are substituted.

The compact asset export screens both Field and Lab training content against Field Dev/Test using exact RGB hashes and DCT pHash distance ≤4. Candidates require resolution before training. Row/group separation and this screen support the split, but cannot establish physical independence or exclude overlap with ImageNet pretraining. Field Test was inspected in earlier experiments, so it remains a fixed historical evaluation rather than a new prospective test.

In [ ]:
# Code cell 5 — Verify assets and configure experiment.
# Choose the architecture and seed budget before initializing an experiment.
ARCHITECTURE = 'resnet18'  # Later: 'vit_small', with a different EXPERIMENT_ID.
PLANNED_SEEDS = (42, 43, 44)  # Use (42,) for a predeclared pilot, before creating the run.
EXPERIMENT_ID = 'controlled_resnet18_s42_43_44_v1'
RESNET_RUN_FOR_VIT = OUTPUT_ROOT / 'controlled_resnet18_s42_43_44_v1'

assets = Assets(REPO, ASSET_ROOT)
RUN = open_experiment(
    REPO, assets, OUTPUT_ROOT, EXPERIMENT_ID, ARCHITECTURE, PLANNED_SEEDS,
    resnet_run=RESNET_RUN_FOR_VIT if ARCHITECTURE == 'vit_small' else None,
)
config = json.loads((RUN / 'config.json').read_text())
display(pd.DataFrame([
    {'Split': name, 'Images': len(frame), 'Role': role}
    for name, role in [('field_adapt_train', 'Field training'), ('lab_train', 'Lab rehearsal'),
                       ('field_dev', 'Checkpoint selection'), ('field_test', 'Post-training evaluation')]
    for frame in [assets.frames[name]]
]))
print('Experiment:', RUN)
print('Initialization SHA-256:', config['initial_sha256'])

## 5. Matched Field exposure

Before training, inspect the deterministic schedule. ResNet18 performs 24 paired updates per epoch and 192 per condition over eight epochs. ViT-Small performs 47 and 376, respectively. Both present every Field example eight times (2,960 presentations per condition); Mixed adds 2,960 Lab presentations. Identical Field tensors are passed to both models, and the actual row IDs, augmentation hashes and optimizer updates are logged.

Selection may choose an earlier shared epoch. Exposure and update totals are checked for both the complete budget and the selected pair. The schedule assertion below samples batches only; it does not train a model.

In [ ]:
# Code cell 6 — Assert matched exposure schedule.
budget = []
for seed in PLANNED_SEEDS:
    counts = np.zeros(370, dtype=int)
    steps = 0
    for epoch in range(1, config['head_epochs']+config['last_block_epochs']+1):
        plan = epoch_plan(config, seed, epoch, len(assets.frames['lab_train']))
        for field_indices, lab_indices in plan:
            assert len(field_indices) == len(lab_indices)
            counts[field_indices] += 1
            steps += 1
    assert np.array_equal(counts, np.full(370, 8))
    budget.append({'Seed': seed, 'Field-only Field presentations': int(counts.sum()),
                   'Mixed Field presentations': int(counts.sum()), 'Updates per condition': steps,
                   'Additional Mixed Lab presentations': int(counts.sum())})
display(pd.DataFrame(budget))

## 6. Controlled fine-tuning

### Stage A — CUDA smoke test

This checks the real baseline, training data, CUDA forward/backward, AdamW update and persistent checkpoint write/read using a disposable model. It never changes the initialization checkpoint. The full experiment always reloads the original baseline.

In [ ]:
# Code cell 7 — Stage A: CUDA smoke test.
smoke = smoke_test(RUN, assets)
assert smoke['passed']
display(pd.DataFrame([{'CUDA smoke test': 'passed', 'GPU': smoke['environment']['device'],
                       'Checkpoint write/read': 'passed'}]))

### Stage B — first declared seed

The next cell starts training. Run seed 42 completely before proceeding. Both conditions are trained as a pair. Each completed epoch is atomically saved to Google Drive with both model states, both optimizer states, exposure counters, histories and shared best-checkpoint state.

After a disconnect, rerun setup with the **same bundle, assets, experiment ID and configuration**, then rerun this cell. A partial epoch is replayed from the last committed epoch, avoiding partial exposure accounting. Complete seeds are verified and skipped. Library-version or configuration changes stop resumption rather than silently alter the experiment.

In [ ]:
# Code cell 8 — Stage B: train/resume seed 42.
first_pair = train_pair(RUN, assets, seed=42)
display(pd.DataFrame([{key: first_pair[key] for key in [
    'seed', 'field_presentations_per_condition', 'optimizer_updates_per_condition',
    'additional_lab_presentations_mixed', 'selected_common_epoch', 'elapsed_pair_seconds']}]))

### Stage C — additional predetermined seeds

Run all declared seeds; do not choose a seed from observed performance. This cell can be rerun safely after a disconnect. For the predeclared one-seed pilot it performs no additional training. Resource limits that prevent completing the declared budget must be reported; the evaluation cell will stop rather than quietly omit a seed.

### Stage D — ViT-Small

Only after the ResNet18 pair(s) validate, start a separate Colab session or rerun the configuration cell with `ARCHITECTURE='vit_small'`, a **new** experiment ID, and `RESNET_RUN_FOR_VIT` pointing to the completed ResNet run. Run the same cells. The baseline is always ViT-Small Lab, never the adapted ResNet model. ResNet outputs remain intact.

In [ ]:
# Code cell 9 — Stage C: remaining seeds and training validation.
for seed in PLANNED_SEEDS:
    if seed != 42:
        train_pair(RUN, assets, seed=seed)
training_validation = validate_training(RUN)
assert training_validation['matched_exposure_and_updates']
completed = pd.DataFrame(training_validation['pairs'])
display(completed[['seed', 'field_presentations_per_condition', 'optimizer_updates_per_condition',
                   'selected_common_epoch', 'selected_field_presentations_per_condition',
                   'selected_updates_per_condition']])

## 7. Validation and checkpoint selection

Learning curves report Field Dev macro-F1 for every fixed epoch. The vertical marker is the pair's shared selected epoch, determined by the prespecified mean score. No Field Test or external model predictions are accessed by training or selection. Full-budget and selected-checkpoint counts are saved separately.

In [ ]:
# Code cell 10 — Plot Field Dev learning curves.
histories = pd.concat([pd.read_csv(RUN / f'seed_{seed}/history.csv') for seed in PLANNED_SEEDS], ignore_index=True)
fig, axes = plt.subplots(1, len(PLANNED_SEEDS), figsize=(5*len(PLANNED_SEEDS), 4), squeeze=False, sharey=True)
for ax, seed in zip(axes[0], PLANNED_SEEDS):
    for condition, color, marker in [('field_only', '#4477AA', 'o'), ('mixed', '#228833', 's')]:
        frame = histories[histories.seed.eq(seed) & histories.condition.eq(condition)]
        ax.plot(frame.epoch, frame.macro_f1, marker=marker, color=color, label=condition.replace('_', ' ').title())
    selected_epoch = int(completed[completed.seed.eq(seed)].selected_common_epoch.iloc[0])
    ax.axvline(selected_epoch, color='#666666', linestyle='--', label=f'Shared selection: {selected_epoch}')
    ax.set(title=f'Seed {seed} — Field Dev (n=92)', xlabel='Completed epoch', ylabel='Macro-F1', ylim=(0, 1))
    ax.legend(fontsize=9)
fig.tight_layout()
(RUN / 'figures').mkdir(exist_ok=True)
fig.savefig(RUN / 'figures/field_dev_learning_curves.png', dpi=160)
plt.show()

## 8. Held-out evaluation

Run this only after the complete declared training budget has validated. It seals the configuration and selected checkpoint hashes, then evaluates the fixed Field Test (47 images). Further training under the same experiment ID is refused after sealing. Repeating evaluation from the same frozen checkpoints is safe.

Accuracy and macro-F1 use the canonical five-class mapping. Seed-level results remain visible; means and sample standard deviations describe this small set of runs, not independent dataset replicates. A one-seed pilot has no run-to-run variability estimate. No external or Healthy-v2 inference is included in this workflow.

In [ ]:
# Code cell 11 — Seal checkpoints and evaluate Field Test.
summary, per_class = evaluate_heldout(RUN, assets)
display(summary[['architecture', 'seed', 'condition', 'correct', 'n', 'accuracy', 'macro_f1']]
        .style.format({'accuracy': '{:.4f}', 'macro_f1': '{:.4f}'}))
aggregate = summary.groupby('condition')[['accuracy', 'macro_f1']].agg(['mean', 'std'])
display(aggregate.round(4))
paired_f1 = summary.pivot(index='seed', columns='condition', values='macro_f1')
paired_f1['Mixed minus Field-only'] = paired_f1.mixed-paired_f1.field_only
display(paired_f1.round(4))

Historical references are a separate, uncontrolled comparison. In Notebooks 06–08, ResNet baseline Field-only / Mixed Field Dev macro-F1 was 0.4119 / 0.3523; geometric ResNet Field-only / Mixed was 0.4347 / 0.4459; ViT-Small was 0.3838 / 0.3872. Their different Field exposure and, for the final geometric comparison, different initialization prevent direct causal interpretation. They are not pooled with the new seed-level Field Test results.

## 9. Per-class error analysis

Report per-class precision and recall, with particular attention to Early Blight, Bacterial Spot and Healthy. The earlier external Early Blight trade-off motivates this analysis, but does not select settings here. Field Test has different provenance and small class denominators; it cannot resolve the earlier external finding by itself.

In [ ]:
# Code cell 12 — Per-class recall.
display(per_class[['seed', 'condition', 'class_name', 'support', 'precision', 'recall']]
        .style.format({'precision': '{:.3f}', 'recall': '{:.3f}'}))
focus = per_class[per_class.class_name.isin(['Early Blight', 'Bacterial Spot', 'Healthy'])]
fig, ax = plt.subplots(figsize=(9, 4.5))
means = focus.groupby(['class_name', 'condition']).recall.mean().unstack().reindex(['Early Blight', 'Bacterial Spot', 'Healthy'])
means.plot.bar(ax=ax, color=['#4477AA', '#228833'], rot=0)
ax.set(ylabel='Mean recall across declared seeds', xlabel='', ylim=(0, 1),
       title='Controlled Field Test class trade-offs (n=47)')
fig.tight_layout()
fig.savefig(RUN / 'figures/field_test_class_recall.png', dpi=160)
plt.show()

In [ ]:
# Code cell 13 — Confusion matrices.
fig, axes = plt.subplots(len(PLANNED_SEEDS), 2, figsize=(11, 4.4*len(PLANNED_SEEDS)), squeeze=False)
for row, seed in enumerate(PLANNED_SEEDS):
    for col, condition in enumerate(CONDITIONS):
        matrix = pd.read_csv(RUN / f'seed_{seed}/confusion_{condition}_field_test.csv', index_col=0).to_numpy()
        ax = axes[row, col]
        ax.imshow(matrix, cmap='Blues', vmin=0, vmax=max(1, matrix.sum(axis=1).max()))
        for i in range(5):
            for j in range(5):
                ax.text(j, i, str(matrix[i, j]), ha='center', va='center',
                        color='white' if matrix[i, j] > matrix.sum(axis=1).max()/2 else '#222222')
        ax.set(xticks=range(5), yticks=range(5), xticklabels=CLASSES, yticklabels=CLASSES,
               title=f'Seed {seed} — {condition.replace("_", " ").title()}', xlabel='Predicted class', ylabel='True class')
        ax.tick_params(axis='x', rotation=40)
fig.tight_layout()
fig.savefig(RUN / 'figures/field_test_confusion_matrices.png', dpi=160)
plt.show()

In [ ]:
# Code cell 14 — Early Blight errors.
early_errors = []
for seed in PLANNED_SEEDS:
    for condition in CONDITIONS:
        frame = pd.read_csv(RUN / f'seed_{seed}/predictions_{condition}_field_test.csv')
        early = frame[frame.label_id.eq(1)]
        for predicted in range(5):
            early_errors.append({'Seed': seed, 'Condition': condition, 'Predicted class': CLASSES[predicted],
                                 'Early Blight images': int(early.predicted_label_id.eq(predicted).sum())})
early_errors = pd.DataFrame(early_errors)
display(early_errors.pivot(index=['Seed', 'Condition'], columns='Predicted class', values='Early Blight images'))
early_errors.to_csv(RUN / 'early_blight_prediction_destinations.csv', index=False)

## 10. Discussion and conclusion

Interpret the executed seed-level and class-level results together. Does the Mixed-minus-Field-only macro-F1 difference have the same direction across seeds? Do Healthy changes accompany losses in Early Blight or Bacterial Spot recall? Lab rehearsal adds information and compute; an improvement would support that intervention under this fixed recipe, not a general architectural claim.

The controls remove unequal Field exposure and update counts. Remaining limits include development-set selection, coupled common-epoch selection, one fixed Lab-loss weight, fixed final-block adaptation, architecture-specific pretraining, small imbalanced Field Test classes, historical test inspection and imperfect duplicate screening. Seed repetitions reuse the same images. They do not create independent test samples.

Conclusions are **pending Colab execution**. Do not insert historical or fabricated numbers in the controlled-result tables. External/Healthy crop findings remain separate exploratory evidence, and no settings may be retuned from their outcomes.

In [ ]:
# Code cell 15 — Save findings.
# Save a concise factual findings file from the executed outputs.
delta = paired_f1['Mixed minus Field-only']
lines = [f'# Controlled {ARCHITECTURE} findings', '',
         f'Declared seeds: {list(PLANNED_SEEDS)}.',
         'Full budget per condition: 2,960 Field presentations over eight epochs.',
         f'Optimizer updates per condition: {int(completed.optimizer_updates_per_condition.iloc[0])}.',
         'Mixed adds 2,960 Lab presentations per seed; computation is not matched.', '',
         'Field Test accuracy / macro-F1 by seed and condition:']
lines += [f'- Seed {r.seed}, {r.condition}: {r.correct}/{r.n}, {r.accuracy:.4f} / {r.macro_f1:.4f}.' for r in summary.itertuples()]
lines += ['', f'Mean paired macro-F1 change (Mixed − Field-only): {delta.mean():+.4f}.',
          'Class trade-offs, small sample sizes and historical test reuse constrain interpretation.',
          'Selected checkpoints share an epoch within each pair; counts are in training_validation.json.']
(RUN / 'findings.md').write_text('\n'.join(lines)+'\n')
print('Persistent results:', RUN)

### Import completed results locally

After Colab finishes, download the complete experiment folder from Drive, including checkpoints, `config.json`, `environment.json`, training validation, prediction CSVs and figures. Copy it into local `outputs/controlled_field_mixed/<same experiment ID>/` without overwriting an existing run. This directory is ignored by Git. Also download the executed Notebook 15 to review its real saved outputs. A minimal import command is in the bundled guide.

Use a separate experiment ID for ViT-Small. A disconnect requires rerunning code cells 1–7 and the relevant training cell; completed seeds are verified and skipped. Keep the exact code bundle and asset manifest associated with each experiment. Device or CUDA changes can affect bitwise reproducibility even when versions match; each smoke test records the current environment.
